# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Marrwan1/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

Ranked actions + reason codes:

Score = (expected_ctr - actual_ctr) * gsc_impressions
Ranked descending — highest opportunity at the top.

Reason codes and their actions:

LOW_CTR_GOOD_POSITION → REFRESH_TITLE_META
  Page ranks in positions 1-10 but CTR is below expected.
  The page is visible — the title or meta description is not compelling enough.
  Action: rewrite title and meta to improve click appeal.

HIGH_IMP_LOW_CTR → REFRESH_CONTENT
  Page has high impressions (≥1000) but CTR is below expected at any position.
  The audience is there but the snippet does not match search intent.
  Action: refresh content and update the meta to match intent.

Archetype → action mapping:
- High impressions + low CTR + good position  → REFRESH_TITLE_META (quick win)
- High impressions + low CTR + poor position  → REFRESH_CONTENT (deeper fix)
- Low impressions + any CTR                   → MONITOR (not enough signal yet)

Decay / refresh insight:
Pages with high days_with_data and declining CTR over time are likely
decaying — they once ranked well but content has gone stale. These are
the highest-value refresh candidates because the page already has
authority; a content update may recover lost clicks without rebuilding
from scratch.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Intended use and limits:

Intended use:
This playbook is a decision-support tool for content teams.
The ranked queue surfaces pages worth a human reviewer's attention —
it does not replace editorial judgment.

Appropriate uses:
- Weekly triage: which pages to review first
- Prioritising a content refresh backlog
- Identifying quick-win title/meta opportunities

Limits:
1. The model was trained on one month (2026-03) and tested on one month
   (2026-04) — it may not generalise to all clients or all seasons.
2. The label (clicks grew ≥20%) does not prove a refresh caused the growth.
   External demand shifts can produce the same signal.
3. Pages with zero GA4 data are scored on GSC signals only — the score
   is less reliable for clients without GA4 connected.
4. The score does not account for editorial cost — a high-score page may
   require more effort than the click gain justifies.
5. This is not a production system — no real-time data, no automated
   publishing, no client-facing output without human review.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Human review + the no-go list:

Human review rules:
- A reviewer must inspect every page in the top 50 before any action is taken.
- The reason code is a starting point, not a verdict — the reviewer decides
  whether the action fits the page's actual content and business context.
- Any page flagged for REFRESH_CONTENT must have its search intent verified
  manually before rewriting begins.

Cost / value thinking:
- REFRESH_TITLE_META: low cost (30–60 min), high potential value for
  high-impression pages — prioritise these first.
- REFRESH_CONTENT: higher cost (2–4 hours), justified only when impressions
  are high enough that a CTR improvement returns meaningful clicks.
- MONITOR: no action cost, revisit next month if signal strengthens.

No-go list — do NOT automate or act on without review:
1. Pages with fewer than 3 clicks in the feature month — signal too weak.
2. Pages published less than 60 days ago — rankings have not settled.
3. Pages where the client has flagged a pending redesign or migration.
4. Pages scoring high only because of a one-week demand spike — check

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

Monitoring / retrain triggers:

Light monitoring:
- Re-run the scoring notebook monthly on the latest closed month.
- Track the share of REFRESH_TITLE_META pages that show CTR improvement
  90 days after the action — this is the playbook's ground truth.

Retrain triggers:
1. AUC on the latest month drops below 0.65 — model has drifted.
2. The positive label rate shifts by more than 5 percentage points
   from the training baseline (16.2%) — label distribution has changed.
3. A new data source becomes available (e.g. GA4 connected for more
   clients) — retrain to include the new signals.
4. More than 3 months have passed since the last training run.

What should NOT be automated:
- The retrain itself — a human must verify the new label definition
  and check for leakage before a new model goes into the queue.
- Any client-facing recommendation — the queue is an internal triage
  tool, not an automated content brief.
- Monitoring alerts — a drop in AUC is a signal to investigate,
  not an automatic rollback trigger.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [2]:
import duckdb
from google.colab import userdata
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import pandas as pd
import os
import json

token = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute(f"CREATE SECRET hf_secret (TYPE huggingface, TOKEN '{token}')")

BASE = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance"
MAR  = f"{BASE}/month=2026-03/*.parquet"
APR  = f"{BASE}/month=2026-04/*.parquet"

# ── Features + label ────────────────────────────────────────────────────────
feat = con.sql(f"""
SELECT client_hash_id, content_hash_id,
    SUM(gsc_impressions)  AS gsc_impressions,
    SUM(gsc_clicks)       AS gsc_clicks,
    AVG(gsc_avg_position) AS gsc_avg_position,
    CASE WHEN SUM(gsc_impressions) = 0 THEN NULL
         ELSE SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions) END AS ctr,
    COUNT(DISTINCT report_date) AS days_with_data,
    SUM(gsc_clicks) AS total_clicks
FROM read_parquet('{MAR}')
WHERE gsc_data_available IS TRUE AND gsc_impressions > 0
GROUP BY client_hash_id, content_hash_id
""").df()

label = con.sql(f"""
SELECT client_hash_id, content_hash_id,
       SUM(gsc_clicks) AS total_clicks_next
FROM read_parquet('{APR}')
WHERE gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
""").df()

df = feat.merge(label, on=["client_hash_id","content_hash_id"], how="inner").dropna()
df["label"] = (df["total_clicks_next"] > df["total_clicks"] * 1.20).astype(int)

# ── Train model ─────────────────────────────────────────────────────────────
FEATURES = ["gsc_impressions", "gsc_clicks", "gsc_avg_position", "ctr", "days_with_data"]
rf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1)
rf.fit(df[FEATURES], df["label"])
df["rf_score"] = rf.predict_proba(df[FEATURES])[:, 1]

# ── Reason code + action ────────────────────────────────────────────────────
df["expected_ctr"] = df["gsc_avg_position"].apply(
    lambda p: 0.0048 if p <= 3 else 0.0035 if p <= 10 else 0.0028 if p <= 20 else 0.0013
)
df["reason_code"] = df.apply(lambda r:
    "LOW_CTR_GOOD_POSITION" if r["gsc_avg_position"] <= 10 and r["ctr"] < r["expected_ctr"]
    else "HIGH_IMP_LOW_CTR"  if r["gsc_impressions"] >= 1000 and r["ctr"] < r["expected_ctr"]
    else "MONITOR", axis=1
)
df["action"] = df["reason_code"].map({
    "LOW_CTR_GOOD_POSITION" : "REFRESH_TITLE_META",
    "HIGH_IMP_LOW_CTR"      : "REFRESH_CONTENT",
    "MONITOR"               : "MONITOR",
})

# ── Ranked queue ────────────────────────────────────────────────────────────
queue = df[df["action"] != "MONITOR"].sort_values("rf_score", ascending=False)

# ── Exports ─────────────────────────────────────────────────────────────────
os.makedirs("work/outputs",  exist_ok=True)
os.makedirs("work/figures",  exist_ok=True)

# CSV
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

# Metrics JSON
auc = roc_auc_score(df["label"], df["rf_score"])
metrics = {
    "model"         : "RandomForest",
    "train_month"   : "2026-03",
    "label_month"   : "2026-04",
    "auc"           : round(auc, 4),
    "queue_size"    : len(queue),
    "positive_rate" : round(df["label"].mean(), 4),
}
with open("work/outputs/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Exports done:")
print(f"  CSV   : work/outputs/baseline_action_score.csv ({len(queue):,} rows)")
print(f"  JSON  : work/outputs/metrics.json")
print(f"  AUC   : {auc:.4f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Exports done:
  CSV   : work/outputs/baseline_action_score.csv (75,869 rows)
  JSON  : work/outputs/metrics.json
  AUC   : 0.7446


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.